
# Information Group 3: Other-Skill Student History

## Objective
Generate historical features using a student's previous questions
whose skills are different from the skills of the current question.

## Feature Engineering Methods
1. Binary other-skill history
2. Fractional other-skill history

## Model Versions
- V3.1: Binary other-skill accuracy
- V3.2: Fractional other-skill accuracy
- V3.3: Binary accuracy + binary experience
- V3.4: Fractional accuracy + fractional experience

## Validation
- Skill relationship correctness
- Chronological global priors
- Student-specific historical counts
- Smoothed accuracy
- Experience features
- Temporal leakage prevention


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

# Locate project root
project_root = Path.cwd().parents[2]

# Make src importable
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.feature_engineering import (
    add_other_skill_history_features
)

print("Imports completed successfully.")

Imports completed successfully.


In [2]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

# Locate project root
project_root = Path.cwd().parents[2]

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.feature_engineering import (
    add_other_skill_history_features
)

print("Group 3 feature engineering function imported successfully.")

Group 3 feature engineering function imported successfully.


In [3]:
# Synthetic dataset for testing Group 3

test_data = pd.DataFrame({
    "user_id": [1, 1, 1, 1],
    "order_id": [1, 2, 3, 4],
    "skill_set": [
        {"A", "B"},
        {"C", "D"},
        {"A", "C", "D"},
        {"A", "B"}
    ],
    "correct": [1, 0, 1, 0]
})

test_data

,user_id,order_id,skill_set,correct
0,1,1,"{A, B}",1
1,1,2,"{C, D}",0
2,1,3,"{C, D, A}",1
3,1,4,"{A, B}",0


In [4]:
test_result = add_other_skill_history_features(
    data=test_data,
    alpha=10
)

# Inspect the features of the target question Q4
test_result.loc[
    test_result["order_id"] == 4,
    [
        "order_id",
        "other_skill_previous_interactions",
        "other_skill_previous_correct",
        "other_skill_global_prior",
        "other_skill_smoothed_accuracy",
        "weighted_other_skill_previous_interactions",
        "weighted_other_skill_previous_correct",
        "weighted_other_skill_global_prior",
        "weighted_other_skill_smoothed_accuracy",
        "log_other_skill_previous_interactions",
        "log_weighted_other_skill_previous_interactions"
    ]
]

,order_id,other_skill_previous_interactions,other_skill_previous_correct,other_skill_global_prior,other_skill_smoothed_accuracy,weighted_other_skill_previous_interactions,weighted_other_skill_previous_correct,weighted_other_skill_global_prior,weighted_other_skill_smoothed_accuracy,log_other_skill_previous_interactions,log_weighted_other_skill_previous_interactions
3,4,1.0,0.0,0.0,0.0,1.666667,0.666667,0.4,0.4,0.693147,0.980829


In [5]:
# Change only the current target's outcome
modified_data = test_data.copy()

modified_data.loc[
    modified_data["order_id"] == 4,
    "correct"
] = 1

modified_result = add_other_skill_history_features(
    data=modified_data,
    alpha=10
)

# Compare original and modified historical features
feature_columns = [
    "other_skill_previous_interactions",
    "other_skill_previous_correct",
    "other_skill_global_prior",
    "other_skill_smoothed_accuracy",
    "weighted_other_skill_previous_interactions",
    "weighted_other_skill_previous_correct",
    "weighted_other_skill_global_prior",
    "weighted_other_skill_smoothed_accuracy"
]

original_features = (
    test_result.loc[test_result["order_id"] == 4, feature_columns]
    .reset_index(drop=True)
)

modified_features = (
    modified_result.loc[
        modified_result["order_id"] == 4,
        feature_columns
    ].reset_index(drop=True)
)

print("Original and modified historical features identical:",
      np.allclose(original_features, modified_features))

Original and modified historical features identical: True


In [6]:
# Add another student's interactions
extended_data = pd.concat(
    [
        test_data,
        pd.DataFrame({
            "user_id": [2, 2],
            "order_id": [5, 6],
            "skill_set": [{"C", "D"}, {"E", "F"}],
            "correct": [1, 1]
        })
    ],
    ignore_index=True
)

extended_result = add_other_skill_history_features(
    data=extended_data,
    alpha=10
)

# Compare student 1's personal history features for Q4
personal_features = [
    "other_skill_previous_interactions",
    "other_skill_previous_correct",
    "weighted_other_skill_previous_interactions",
    "weighted_other_skill_previous_correct"
]

original_q4 = test_result.loc[
    test_result["order_id"] == 4,
    personal_features
].reset_index(drop=True)

extended_q4 = extended_result.loc[
    (extended_result["user_id"] == 1) &
    (extended_result["order_id"] == 4),
    personal_features
].reset_index(drop=True)

print(
    "Student 1's personal history unchanged:",
    np.allclose(original_q4, extended_q4)
)

Student 1's personal history unchanged: True


In [7]:
# Simulate separate training and validation data

train_data = test_data.iloc[:2].copy()

validation_data = test_data.iloc[2:].copy()

# Calculate validation features using training data for the global prior
validation_result_1 = add_other_skill_history_features(
    data=validation_data,
    alpha=10,
    prior_data=train_data
)

# Change validation outcomes
modified_validation = validation_data.copy()
modified_validation["correct"] = 1 - modified_validation["correct"]

validation_result_2 = add_other_skill_history_features(
    data=modified_validation,
    alpha=10,
    prior_data=train_data
)

# Compare global priors
prior_columns = [
    "other_skill_global_prior",
    "weighted_other_skill_global_prior"
]

print(
    "Global priors unchanged after modifying validation outcomes:",
    np.allclose(
        validation_result_1[prior_columns],
        validation_result_2[prior_columns]
    )
)

Global priors unchanged after modifying validation outcomes: True


In [8]:
# Add a future interaction
future_data = pd.concat(
    [
        test_data,
        pd.DataFrame({
            "user_id": [1],
            "order_id": [5],
            "skill_set": [{"C", "D"}],
            "correct": [1]
        })
    ],
    ignore_index=True
)

future_result = add_other_skill_history_features(
    data=future_data,
    alpha=10
)

# Compare Q4 features before and after adding Q5
q4_original = test_result.loc[
    test_result["order_id"] == 4,
    feature_columns
].reset_index(drop=True)

q4_after_future = future_result.loc[
    future_result["order_id"] == 4,
    feature_columns
].reset_index(drop=True)

print(
    "Q4 historical features unchanged after adding Q5:",
    np.allclose(q4_original, q4_after_future)
)

Q4 historical features unchanged after adding Q5: True



## Synthetic Validation Results

The Group 3 feature engineering function passed five validation tests:

1. Binary and fractional mathematical correctness.
2. Temporal leakage prevention.
3. Student-level history isolation.
4. Training-validation global-prior isolation.
5. Chronological future-interaction isolation.

The implementation is ready for full-dataset feature generation.


In [9]:
# Load cleaned interactions
data_path = (
    project_root
    / "data"
    / "processed"
    / "main_questions_cleaned.csv"
)

split_path = (
    project_root
    / "data"
    / "processed"
    / "split_assignments.csv"
)

main_df = pd.read_csv(data_path)
split_df = pd.read_csv(split_path)

# Attach each student's partition and validation fold
main_df = main_df.merge(
    split_df,
    on="user_id",
    how="left",
    validate="many_to_one"
)

print("Dataset shape:", main_df.shape)
print("\nPartition distribution:")
print(main_df["partition"].value_counts())

print("\nValidation fold distribution:")
print(main_df.groupby(
    ["partition", "validation_fold"],
    dropna=False
).size())

print("\nSkill-set column type:", type(main_df["skill_set"].iloc[0]))
print("Example skill_set:", repr(main_df["skill_set"].iloc[0]))

Dataset shape: (275458, 9)

Partition distribution:
partition
train_pool    219983
test           55475
Name: count, dtype: int64

Validation fold distribution:
partition   validation_fold
test        NaN                55475
train_pool  1.0                42659
            2.0                43710
            3.0                49797
            4.0                41672
            5.0                42145
dtype: int64

Skill-set column type: <class 'str'>
Example skill_set: "{'70', '2', '37'}"


In [10]:

import ast

def _normalise_other_skill_set(value):
    """Convert stored skill information into a Python set."""

    if isinstance(value, (set, frozenset)):
        return set(value)

    if value is None:
        return set()

    try:
        if pd.isna(value):
            return set()
    except (TypeError, ValueError):
        pass

    if isinstance(value, str):
        value = value.strip()

        # Parse CSV representations such as "{'70', '2', '37'}"
        try:
            parsed = ast.literal_eval(value)

            if isinstance(parsed, (set, list, tuple)):
                return set(str(skill) for skill in parsed)

        except (ValueError, SyntaxError):
            pass

        # Support original skill_id strings such as "1_13"
        return set(value.split("_"))

    return set()


In [11]:
from src.feature_engineering import _normalise_other_skill_set

sample = main_df["skill_set"].iloc[0]

print("Original:", sample)
print("Parsed:", _normalise_other_skill_set(sample))
print("Number of skills:", len(_normalise_other_skill_set(sample)))

Original: {'70', '2', '37'}
Parsed: {"{'70', '2', '37'}"}
Number of skills: 1


In [12]:
import importlib
import src.feature_engineering as fe

# Reload the module from the updated Python file
importlib.reload(fe)

# Test the updated helper
sample = main_df["skill_set"].iloc[0]

parsed_skills = fe._normalise_other_skill_set(sample)

print("Original:", repr(sample))
print("Parsed:", parsed_skills)
print("Number of skills:", len(parsed_skills))

Original: "{'70', '2', '37'}"
Parsed: {"{'70', '2', '37'}"}
Number of skills: 1


In [13]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import importlib

# Locate project root
project_root = Path.cwd().parents[2]

# Add project root to Python's module search path
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Import reusable modules
from src.data_loader import load_cv_data
import src.feature_engineering as fe

# Reload the latest implementation
importlib.reload(fe)

print("Project root:", project_root)
print("Setup completed.")

Project root: c:\Users\rajur\OneDrive\Desktop\next-move-ai
Setup completed.


In [14]:
# Load five cross-validation fold datasets
fold_datasets = load_cv_data(project_root)

# Convert skill IDs into Python sets
def convert_skill_id(skill_id):
    if pd.isna(skill_id):
        return set()

    return set(str(skill_id).split("_"))


for train_df, val_df in fold_datasets:

    train_df["skill_set"] = train_df["skill_id"].apply(
        convert_skill_id
    )

    val_df["skill_set"] = val_df["skill_id"].apply(
        convert_skill_id
    )


# Display fold sizes
for fold_number, (train_df, val_df) in enumerate(
    fold_datasets, start=1
):
    print(
        f"Fold {fold_number}: "
        f"Training = {train_df.shape}, "
        f"Validation = {val_df.shape}"
    )


# Inspect skill conversion
fold_train_df, fold_val_df = fold_datasets[0]

print("\nOriginal skill IDs:")
print(fold_train_df["skill_id"].head(10).tolist())

print("\nConverted skill sets:")
print(fold_train_df["skill_set"].head(10).tolist())

print("\nSkill-set data type:")
print(type(fold_train_df["skill_set"].iloc[0]))

Fold 1: Training = (177324, 7), Validation = (42659, 7)
Fold 2: Training = (176273, 7), Validation = (43710, 7)
Fold 3: Training = (170186, 7), Validation = (49797, 7)
Fold 4: Training = (178311, 7), Validation = (41672, 7)
Fold 5: Training = (177838, 7), Validation = (42145, 7)

Original skill IDs:
['2_37_70', '2_37_70', '2_37_70', '2_37_70', '2_37_70', '2_37_70', '2_37_48_77', '2_48_79', '2_37_48', '2_48_79']

Converted skill sets:
[{'70', '37', '2'}, {'70', '37', '2'}, {'70', '37', '2'}, {'70', '37', '2'}, {'70', '37', '2'}, {'70', '37', '2'}, {'48', '77', '37', '2'}, {'48', '2', '79'}, {'48', '37', '2'}, {'48', '2', '79'}]

Skill-set data type:
<class 'set'>


In [15]:
# Select a sample from Fold 1
sample_data = fold_train_df.head(5000).copy()

# Generate Group 3 features
sample_features = fe.add_other_skill_history_features(
    data=sample_data,
    alpha=10,
    prior_data=fold_train_df
)

# Inspect generated features
display(
    sample_features[
        [
            "user_id",
            "order_id",
            "skill_id",
            "correct",
            "other_skill_previous_interactions",
            "other_skill_previous_correct",
            "other_skill_global_prior",
            "other_skill_smoothed_accuracy",
            "weighted_other_skill_previous_interactions",
            "weighted_other_skill_previous_correct",
            "weighted_other_skill_global_prior",
            "weighted_other_skill_smoothed_accuracy"
        ]
    ].head(15)
)

,user_id,order_id,skill_id,correct,other_skill_previous_interactions,other_skill_previous_correct,other_skill_global_prior,other_skill_smoothed_accuracy,weighted_other_skill_previous_interactions,weighted_other_skill_previous_correct,weighted_other_skill_global_prior,weighted_other_skill_smoothed_accuracy
0,14,21617623,2_37_70,0,0.0,0.0,0.739335,0.739335,0.000000,0.000000,0.738935,0.738935
1,14,21617632,2_37_70,1,0.0,0.0,0.739335,0.739335,0.000000,0.000000,0.738935,0.738935
2,14,21617641,2_37_70,0,0.0,0.0,0.739335,0.739335,0.000000,0.000000,0.738935,0.738935
3,14,21617650,2_37_70,0,0.0,0.0,0.739335,0.739335,0.000000,0.000000,0.738935,0.738935
4,14,21617659,2_37_70,0,0.0,0.0,0.739335,0.739335,0.000000,0.000000,0.738935,0.738935
5,14,21617667,2_37_70,0,0.0,0.0,0.739335,0.739335,0.000000,0.000000,0.738935,0.738935
6,14,21617675,2_37_48_77,0,0.0,0.0,0.739012,0.739012,2.000000,0.333333,0.738900,0.643528
7,14,21617692,2_48_79,0,0.0,0.0,0.739545,0.739545,4.500000,0.666667,0.739264,0.555814
8,14,21617731,2_37_48,1,0.0,0.0,0.738536,0.738536,2.583333,0.333333,0.738370,0.613274
9,14,21617749,2_48_79,0,0.0,0.0,0.739545,0.739545,4.833333,1.000000,0.739275,0.565803


In [16]:
# Features to validate
feature_columns = [
    "other_skill_previous_interactions",
    "other_skill_previous_correct",
    "other_skill_global_prior",
    "other_skill_smoothed_accuracy",
    "weighted_other_skill_previous_interactions",
    "weighted_other_skill_previous_correct",
    "weighted_other_skill_global_prior",
    "weighted_other_skill_smoothed_accuracy",
    "log_other_skill_previous_interactions",
    "log_weighted_other_skill_previous_interactions"
]

print("Missing values:")
print(sample_features[feature_columns].isna().sum())

print("\nBinary count violations:")
print(
    (
        sample_features["other_skill_previous_correct"]
        > sample_features["other_skill_previous_interactions"]
    ).sum()
)

print("\nFractional count violations:")
print(
    (
        sample_features["weighted_other_skill_previous_correct"]
        > sample_features["weighted_other_skill_previous_interactions"]
    ).sum()
)

print("\nBinary global prior range:")
print(
    sample_features["other_skill_global_prior"].min(),
    sample_features["other_skill_global_prior"].max()
)

print("\nFractional global prior range:")
print(
    sample_features["weighted_other_skill_global_prior"].min(),
    sample_features["weighted_other_skill_global_prior"].max()
)

print("\nBinary smoothed accuracy range:")
print(
    sample_features["other_skill_smoothed_accuracy"].min(),
    sample_features["other_skill_smoothed_accuracy"].max()
)

print("\nFractional smoothed accuracy range:")
print(
    sample_features["weighted_other_skill_smoothed_accuracy"].min(),
    sample_features["weighted_other_skill_smoothed_accuracy"].max()
)

Missing values:
other_skill_previous_interactions                 0
other_skill_previous_correct                      0
other_skill_global_prior                          0
other_skill_smoothed_accuracy                     0
weighted_other_skill_previous_interactions        0
weighted_other_skill_previous_correct             0
weighted_other_skill_global_prior                 0
weighted_other_skill_smoothed_accuracy            0
log_other_skill_previous_interactions             0
log_weighted_other_skill_previous_interactions    0
dtype: int64

Binary count violations:
0

Fractional count violations:
0

Binary global prior range:
0.5 0.7708333333333334

Fractional global prior range:
0.5 0.7708333333333334

Binary smoothed accuracy range:
0.3307403852592576 0.8933319973949202

Fractional smoothed accuracy range:
0.3321975631224362 0.8929570373380444


In [17]:
# Generate Group 3 features for all five folds

v3_fold_datasets = []

for fold_number, (fold_train_df, fold_val_df) in enumerate(
    fold_datasets, start=1
):

    print(f"\nProcessing Fold {fold_number}...")

    # Generate training features
    v3_train_df = fe.add_other_skill_history_features(
        data=fold_train_df,
        alpha=10,
        prior_data=fold_train_df
    )

    # Generate validation features
    v3_val_df = fe.add_other_skill_history_features(
        data=fold_val_df,
        alpha=10,
        prior_data=fold_train_df
    )

    v3_fold_datasets.append(
        (v3_train_df, v3_val_df)
    )

    print("Training features:", v3_train_df.shape)
    print("Validation features:", v3_val_df.shape)

print("\nGroup 3 feature generation completed.")


Processing Fold 1...
Training features: (177324, 17)
Validation features: (42659, 17)

Processing Fold 2...
Training features: (176273, 17)
Validation features: (43710, 17)

Processing Fold 3...
Training features: (170186, 17)
Validation features: (49797, 17)

Processing Fold 4...
Training features: (178311, 17)
Validation features: (41672, 17)

Processing Fold 5...
Training features: (177838, 17)
Validation features: (42145, 17)

Group 3 feature generation completed.


In [18]:
from pathlib import Path

# Base directory for Group 3 engineered features
base_dir = (
    project_root
    / "data"
    / "processed"
    / "engineered_features"
    / "group_3_other_skill_history"
)

# Define the features required by each version
version_columns = {
    "v3.1": [
        "other_skill_smoothed_accuracy"
    ],
    "v3.2": [
        "weighted_other_skill_smoothed_accuracy"
    ],
    "v3.3": [
        "other_skill_smoothed_accuracy",
        "log_other_skill_previous_interactions"
    ],
    "v3.4": [
        "weighted_other_skill_smoothed_accuracy",
        "log_weighted_other_skill_previous_interactions"
    ]
}

# Common columns required by every version
common_columns = [
    "user_id",
    "order_id",
    "correct"
]

# Save each version separately
for version, feature_columns in version_columns.items():

    feature_dir = base_dir / version
    feature_dir.mkdir(parents=True, exist_ok=True)

    columns_to_save = common_columns + feature_columns

    print(f"\nSaving {version}...")

    for fold_number, (train_df, val_df) in enumerate(
        v3_fold_datasets, start=1
    ):

        train_df[columns_to_save].to_csv(
            feature_dir / f"fold_{fold_number}_train.csv",
            index=False
        )

        val_df[columns_to_save].to_csv(
            feature_dir / f"fold_{fold_number}_validation.csv",
            index=False
        )

        print(f"Fold {fold_number} saved.")

    print(f"{version} completed.")

print("\nAll Group 3 feature versions saved successfully!")


Saving v3.1...
Fold 1 saved.
Fold 2 saved.
Fold 3 saved.
Fold 4 saved.
Fold 5 saved.
v3.1 completed.

Saving v3.2...
Fold 1 saved.
Fold 2 saved.
Fold 3 saved.
Fold 4 saved.
Fold 5 saved.
v3.2 completed.

Saving v3.3...
Fold 1 saved.
Fold 2 saved.
Fold 3 saved.
Fold 4 saved.
Fold 5 saved.
v3.3 completed.

Saving v3.4...
Fold 1 saved.
Fold 2 saved.
Fold 3 saved.
Fold 4 saved.
Fold 5 saved.
v3.4 completed.

All Group 3 feature versions saved successfully!


In [19]:
for version in version_columns:

    feature_dir = base_dir / version

    print(f"\n{version}:")

    for fold_number in range(1, 6):

        train_path = feature_dir / f"fold_{fold_number}_train.csv"
        val_path = feature_dir / f"fold_{fold_number}_validation.csv"

        train_df = pd.read_csv(train_path)
        val_df = pd.read_csv(val_path)

        print(
            f"Fold {fold_number}: "
            f"Train {train_df.shape}, "
            f"Validation {val_df.shape}"
        )


v3.1:
Fold 1: Train (177324, 4), Validation (42659, 4)
Fold 2: Train (176273, 4), Validation (43710, 4)
Fold 3: Train (170186, 4), Validation (49797, 4)
Fold 4: Train (178311, 4), Validation (41672, 4)
Fold 5: Train (177838, 4), Validation (42145, 4)

v3.2:
Fold 1: Train (177324, 4), Validation (42659, 4)
Fold 2: Train (176273, 4), Validation (43710, 4)
Fold 3: Train (170186, 4), Validation (49797, 4)
Fold 4: Train (178311, 4), Validation (41672, 4)
Fold 5: Train (177838, 4), Validation (42145, 4)

v3.3:
Fold 1: Train (177324, 5), Validation (42659, 5)
Fold 2: Train (176273, 5), Validation (43710, 5)
Fold 3: Train (170186, 5), Validation (49797, 5)
Fold 4: Train (178311, 5), Validation (41672, 5)
Fold 5: Train (177838, 5), Validation (42145, 5)

v3.4:
Fold 1: Train (177324, 5), Validation (42659, 5)
Fold 2: Train (176273, 5), Validation (43710, 5)
Fold 3: Train (170186, 5), Validation (49797, 5)
Fold 4: Train (178311, 5), Validation (41672, 5)
Fold 5: Train (177838, 5), Validation (42